## Sanity Check: ResiDual ≡ Classic CLAP quando variance_threshold=1.0

Con variance_threshold=1.0 la PCA mantiene **tutte** le componenti principali
(k = embed_dim) e λ viene inizializzato a 1 per tutte le PC.
La trasformazione RD(X, λ) diventa quindi:

  RD(X, 1) = Φᵀ · I · Φ · (X - μ) + μ = X

In [1]:
import torch
import torch.nn.functional as F
import numpy as np

from CLAPWrapper import CLAPWrapper
from datasets.tinysol import TinySOL

DATASET_ROOT    = "../data"
N_SAMPLES       = 200
BATCH_SIZE      = 16
PCA_SAMPLES     = 200

RESIDUAL_CONFIG = {
    'target_layers':      [0, 1, 2, 3],
    'variance_threshold': 1.0,
}

# ============================================================================
# Caricamento modelli
# ============================================================================

print("Caricamento CLAP standard...", end=' ')
clap_standard = CLAPWrapper(
    version  = '2023',
    use_cuda = torch.cuda.is_available(),
    type     = 'classic'
)
print('OK')

print("Caricamento ResiDual CLAP (variance_threshold=1.0, layers 0-3)...", end=' ')
clap_residual = CLAPWrapper(
    version         = '2023',
    use_cuda        = torch.cuda.is_available(),
    type            = 'residual',
    residual_config = RESIDUAL_CONFIG
)
print('OK')

Caricamento CLAP standard... OK
Caricamento ResiDual CLAP (variance_threshold=1.0, layers 0-3)... OK


In [2]:
# ============================================================================
# Dataset — primi N_SAMPLES audio path
# ============================================================================

dataset     = TinySOL(DATASET_ROOT)
audio_paths = []
for i in range(len(dataset)):
    audio_path, _, _ = dataset[i]
    audio_paths.append(audio_path)
    if len(audio_paths) >= N_SAMPLES:
        break

print(f"Campioni selezionati: {len(audio_paths)}")

Dataset già presente in ../data/TinySOL
Loading audio files


100%|██████████| 2913/2913 [00:00<00:00, 9365.00it/s] 

✓ Cache di validazione trovata (2026-04-01T23:54:22.189250): 2913 validi, 0 corrotti. Skip validazione.
Campioni selezionati: 200


### Estrazione embedding PRIMA del fit PCA (baseline pre-fit)

In [3]:
emb_standard = []
emb_pre_fit  = []

for start in range(0, len(audio_paths), BATCH_SIZE):
    batch_paths = audio_paths[start:start + BATCH_SIZE]
    with torch.no_grad():
        e_std = clap_standard.get_audio_embeddings(batch_paths, resample=True)
        e_pre = clap_residual.get_audio_embeddings(batch_paths, resample=True)
    emb_standard.append(e_std.cpu())
    emb_pre_fit.append(e_pre.cpu())

    print(f"  Batch {start//BATCH_SIZE + 1} | std norm={e_std.norm(dim=-1).mean():.4f} "
          f"| res norm={e_pre.norm(dim=-1).mean():.4f}")

emb_standard = torch.cat(emb_standard, dim=0)   # [N, d_proj]
emb_pre_fit  = torch.cat(emb_pre_fit,  dim=0)   # [N, d_proj]

  Batch 1 | std norm=32.1128 | res norm=32.1128
  Batch 2 | std norm=32.1230 | res norm=32.1230
  Batch 3 | std norm=32.1260 | res norm=32.1260
  Batch 4 | std norm=32.1079 | res norm=32.1079
  Batch 5 | std norm=32.1003 | res norm=32.1003
  Batch 6 | std norm=32.1007 | res norm=32.1007
  Batch 7 | std norm=32.1112 | res norm=32.1112
  Batch 8 | std norm=32.1329 | res norm=32.1329
  Batch 9 | std norm=32.1440 | res norm=32.1440
  Batch 10 | std norm=32.1360 | res norm=32.1360
  Batch 11 | std norm=32.1424 | res norm=32.1424
  Batch 12 | std norm=32.1449 | res norm=32.1449
  Batch 13 | std norm=32.1291 | res norm=32.1291


In [4]:
# ============================================================================
# Fit PCA con variance_threshold=1.0 su tutti i layer
# ============================================================================

print("\nFit PCA (variance_threshold=1.0) su tutti i layer...")

# SimpleAudioDataset restituisce tensori audio grezzi (senza label)
class SimpleAudioDataset(torch.utils.data.Dataset):
    def __init__(self, wrapper, audio_paths):
        self.wrapper     = wrapper
        self.audio_paths = audio_paths

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        audio = self.wrapper.load_audio_into_tensor(
            self.audio_paths[idx],
            self.wrapper.args.duration,
            resample=True
        )
        return audio.squeeze()

pca_dataset = SimpleAudioDataset(clap_residual, audio_paths)
pca_loader  = torch.utils.data.DataLoader(pca_dataset, batch_size=BATCH_SIZE, shuffle=False)

fit_info = clap_residual.clap.fit_spectral_components(pca_loader, max_samples=PCA_SAMPLES)


Fit PCA (variance_threshold=1.0) su tutti i layer...


[ResiDual] Raccolta per PCA:   0%|          | 0/13 [00:00<?, ?it/s]

[ResiDual] Raccolti 200 campioni
  layer_0 | block 0 | head_0: k=24 | var@k=1.000
  layer_0 | block 0 | head_1: k=24 | var@k=1.000
  layer_0 | block 0 | head_2: k=24 | var@k=1.000
  layer_0 | block 0 | head_3: k=24 | var@k=1.000
  layer_0 | block 1 | head_0: k=24 | var@k=1.000
  layer_0 | block 1 | head_1: k=24 | var@k=1.000
  layer_0 | block 1 | head_2: k=24 | var@k=1.000
  layer_0 | block 1 | head_3: k=24 | var@k=1.000
  layer_1 | block 0 | head_0: k=24 | var@k=1.000
  layer_1 | block 0 | head_1: k=24 | var@k=1.000
  layer_1 | block 0 | head_2: k=24 | var@k=1.000
  layer_1 | block 0 | head_3: k=24 | var@k=1.000
  layer_1 | block 0 | head_4: k=24 | var@k=1.000
  layer_1 | block 0 | head_5: k=24 | var@k=1.000
  layer_1 | block 0 | head_6: k=24 | var@k=1.000
  layer_1 | block 0 | head_7: k=24 | var@k=1.000
  layer_1 | block 1 | head_0: k=24 | var@k=1.000
  layer_1 | block 1 | head_1: k=24 | var@k=1.000
  layer_1 | block 1 | head_2: k=24 | var@k=1.000
  layer_1 | block 1 | head_3: k=24 |

In [6]:
# ============================================================================
# Verifica post-fit: k == embed_dim per ogni head (variance_threshold=1.0)
# ============================================================================

htsat = clap_residual.clap.audio_encoder.base.htsat

print("\nVerifica k == embed_dim per ogni layer/block/head:")
all_full_rank = []

for layer_name, layer_modules in htsat.spectral_layers.items():
    for block_idx, block_modules in enumerate(layer_modules):
        for head_idx, layer in enumerate(block_modules):
            k         = int((layer.lambda_weights != 0).sum().item())
            embed_dim = layer.embed_dim
            full_rank = (k == embed_dim)
            all_full_rank.append(full_rank)
            status = "✓" if full_rank else "✗"
            print(f"  {status} {layer_name} | block {block_idx} | head {head_idx} "
                  f"→ k={k} / embed_dim={embed_dim}")

assert all(all_full_rank), "Alcuni head non hanno k == embed_dim — unexpected con threshold=1.0!"
print("\n✓ Tutti i head hanno k == embed_dim (λ = 1 per tutte le PC)")


Verifica k == embed_dim per ogni layer/block/head:
  ✓ layer_0 | block 0 | head 0 → k=24 / embed_dim=24
  ✓ layer_0 | block 0 | head 1 → k=24 / embed_dim=24
  ✓ layer_0 | block 0 | head 2 → k=24 / embed_dim=24
  ✓ layer_0 | block 0 | head 3 → k=24 / embed_dim=24
  ✓ layer_0 | block 1 | head 0 → k=24 / embed_dim=24
  ✓ layer_0 | block 1 | head 1 → k=24 / embed_dim=24
  ✓ layer_0 | block 1 | head 2 → k=24 / embed_dim=24
  ✓ layer_0 | block 1 | head 3 → k=24 / embed_dim=24
  ✓ layer_1 | block 0 | head 0 → k=24 / embed_dim=24
  ✓ layer_1 | block 0 | head 1 → k=24 / embed_dim=24
  ✓ layer_1 | block 0 | head 2 → k=24 / embed_dim=24
  ✓ layer_1 | block 0 | head 3 → k=24 / embed_dim=24
  ✓ layer_1 | block 0 | head 4 → k=24 / embed_dim=24
  ✓ layer_1 | block 0 | head 5 → k=24 / embed_dim=24
  ✓ layer_1 | block 0 | head 6 → k=24 / embed_dim=24
  ✓ layer_1 | block 0 | head 7 → k=24 / embed_dim=24
  ✓ layer_1 | block 1 | head 0 → k=24 / embed_dim=24
  ✓ layer_1 | block 1 | head 1 → k=24 / embed_d

### Estrazione embedding DOPO il fit PCA

In [9]:
emb_post_fit = []

for start in range(0, len(audio_paths), BATCH_SIZE):
    batch_paths = audio_paths[start:start + BATCH_SIZE]
    with torch.no_grad():
        e_post = clap_residual.get_audio_embeddings(batch_paths, resample=True)
    emb_post_fit.append(e_post.cpu())

    print(f"  Batch {start//BATCH_SIZE + 1} "
          f"| res norm={e_post.norm(dim=-1).mean():.4f}")

emb_post_fit = torch.cat(emb_post_fit, dim=0)   # [N, d_proj]

  Batch 1 | res norm=32.1128
  Batch 2 | res norm=32.1230
  Batch 3 | res norm=32.1260
  Batch 4 | res norm=32.1079
  Batch 5 | res norm=32.1003
  Batch 6 | res norm=32.1007
  Batch 7 | res norm=32.1112
  Batch 8 | res norm=32.1329
  Batch 9 | res norm=32.1440
  Batch 10 | res norm=32.1360
  Batch 11 | res norm=32.1424
  Batch 12 | res norm=32.1449
  Batch 13 | res norm=32.1291


In [11]:
# ============================================================================
# Confronto 1: pre-fit vs standard  (atteso: identici — stesso del test precedente)
# ============================================================================

def print_comparison(label_a, label_b, emb_a, emb_b):
    abs_diff  = (emb_a - emb_b).abs()
    max_diff  = abs_diff.max().item()
    mean_diff = abs_diff.mean().item()
    cos_sim   = F.cosine_similarity(emb_a, emb_b, dim=-1)
    are_close = torch.allclose(emb_a, emb_b, atol=1e-5)
    verdict   = "✓ PASS" if are_close else "✗ FAIL"

    print(f"\n  {label_a}  vs  {label_b}")
    print(f"  {'─'*50}")
    print(f"  Max  |Δ|          : {max_diff:.2e}")
    print(f"  Mean |Δ|          : {mean_diff:.2e}")
    print(f"  Cosine sim        : min={cos_sim.min():.6f}  mean={cos_sim.mean():.6f}")
    print(f"  torch.allclose    : {are_close}  (atol=1e-5)")
    print(f"  Esito             : {verdict}")

print("=" * 55)
print("  Confronto embedding")
print("=" * 55)

# Pre-fit vs standard: devono essere identici (is_fitted=False → pass-through)
print_comparison("Classic", "ResiDual pre-fit", emb_standard, emb_pre_fit)

# Post-fit vs standard: RD(X,1) = X - μ, ci aspettiamo una piccola differenza
print_comparison("Classic", "ResiDual post-fit (threshold=1.0)", emb_standard, emb_post_fit)

# Pre-fit vs post-fit: quantifica l'effetto della sola sottrazione di μ
print_comparison("ResiDual pre-fit", "ResiDual post-fit", emb_pre_fit, emb_post_fit)

print("\n" + "=" * 55)
print("  Interpretazione attesa")
print("=" * 55)
print("  Classic vs pre-fit  → identici  (pass-through)")
print("  Classic vs post-fit → piccola Δ sistematica")
print("  pre-fit vs post-fit → stessa piccola Δ")

  Confronto embedding

  Classic  vs  ResiDual pre-fit
  ──────────────────────────────────────────────────
  Max  |Δ|          : 0.00e+00
  Mean |Δ|          : 0.00e+00
  Cosine sim        : min=1.000000  mean=1.000000
  torch.allclose    : True  (atol=1e-5)
  Esito             : ✓ PASS

  Classic  vs  ResiDual post-fit (threshold=1.0)
  ──────────────────────────────────────────────────
  Max  |Δ|          : 2.03e-06
  Mean |Δ|          : 2.73e-07
  Cosine sim        : min=1.000000  mean=1.000000
  torch.allclose    : True  (atol=1e-5)
  Esito             : ✓ PASS

  ResiDual pre-fit  vs  ResiDual post-fit
  ──────────────────────────────────────────────────
  Max  |Δ|          : 2.03e-06
  Mean |Δ|          : 2.73e-07
  Cosine sim        : min=1.000000  mean=1.000000
  torch.allclose    : True  (atol=1e-5)
  Esito             : ✓ PASS

  Interpretazione attesa
  Classic vs pre-fit  → identici  (pass-through)
  Classic vs post-fit → piccola Δ sistematica
  pre-fit vs post-fit → stess